# Two-Arm Bandit: Latent Space Visualisation

Loads a pre-trained BundleNet run (from a grid search) and visualises the 3-D latent
trajectories from six canonical viewpoints plus one isometric view.  
Also includes basic dataset sanity checks: sparsity, state transitions, and unique behavioural states.


In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image

from ncmcm.data_loaders.bandit_task import BanditTaskNeuroPixelsDataset
from ncmcm.visualisers.latent_space import LatentSpaceVisualiser


## Configuration

Set the session and grid-search run paths here before running the notebook.


In [ ]:
# --- paths ---
data_path = Path("./JPAS_0023_20230922") 
run_dir = Path("/home/kerim/Projects/Neural Algorithms/NC-MCM/datasets/raw/twoArmBandit/results/archive_20260212/?_grid_search_20260107_005655/?_run_000_data_path=JPAS_0023_20230922_downsample_fs=25_downsample_method=gaussian_good_neurons_only=False_apply_hold_transitions=False_normalize_method=minmax_window=50_latent_dim=3_batch_size=50_learning_rate=0.0001_gamma=0.75")
figures_dir = Path("./figures") / run_dir.name
figures_dir.mkdir(parents=True, exist_ok=True)


# --- figure crop margin (cm on each side) ---
crop_cm = 4


## Latent Space Visualisation

Load the latent trajectories saved by the grid-search run and render the 3-D phase-space
from six canonical viewpoints (front/back/top/bottom/left/right) plus one isometric view.
Each figure is saved as a PNG and cropped by `crop_cm` centimetres on every side to remove whitespace.


In [ ]:
# Load latent trajectories and matching behaviour labels
y = np.load(run_dir / "data/latent_trajectories_train.npy")  # (T, latent_dim)
b = np.load(run_dir / "data/behavior_labels_train.npy")      # (T,)

# Load colour map from a fresh (minimal) dataset instance
ds_colors = BanditTaskNeuroPixelsDataset(
    data_path=data_path,
    downsample_fs=30,
    downsample_method="count",
    good_neurons_only=True,
    normalize_method=None,
)
b_labels = ds_colors.b_labels
b_colors_rgb = ds_colors.get_rgb_colors_for_visualizer()
del ds_colors

vis = LatentSpaceVisualiser(y, b, b_labels, show_points=True, colors=b_colors_rgb, legend=False)

views = [
    ((0, 180), "back"),
    ((-90, 0), "bottom"),
    ((0, 0),   "front"),
    ((90, 0),  "top"),
    ((0, 90),  "right"),
    ((0, -90), "left"),
    ((30, 45), "iso"),
]

for (elev, azim), name in views:
    fname = figures_dir / f"latent_space_{name}.png"
    fig, ax = vis.plot_phase_space(show_fig=False, axis_view=(elev, azim), filename=str(fname))

    with Image.open(fname) as im:
        dpi = im.info.get("dpi", (100, 100))[0]
        pad_px = int(round((crop_cm / 2.54) * dpi))
        w, h = im.size
        box = (pad_px, pad_px, w - pad_px, h - pad_px)
        if box[2] > box[0] and box[3] > box[1]:
            im.crop(box).save(fname)
            print(f"Saved {fname.name}  ({box[2]-box[0]}×{box[3]-box[1]} px)")
        else:
            print(f"Skipping {fname.name}: crop margin too large")
